# All-model held-out evaluation

Computes overall and subgroup test-set metrics used in the model comparison tables.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:

import os
import numpy as np
import pandas as pd
import torch
import joblib

from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset

from transformers import (
    LongformerTokenizer,
    LongformerForSequenceClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


# ============================================================
# CONFIG
# ============================================================

DATA_PATH = str(DATA_DIR / 'GEP_test_80_20.csv')
RESULT_DIR = str(RESULTS_DIR) + os.sep
os.makedirs(RESULT_DIR, exist_ok=True)

BATCH_SIZE_LONGFORMER = 48
BATCH_SIZE_BERT = 4
NUM_WORKERS = 0
PIN_MEMORY = False

DOC_MAX_TOKENS = 4096
CHUNK_SIZE = 510
MAX_LENGTH = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mimic_test = pd.read_csv(DATA_PATH)
mimic_test["text"] = mimic_test["text"].astype(str)
mimic_test["GEP"] = mimic_test["GEP"].astype(int)

# ============================================================
# MODEL CONFIGS
# ============================================================

EVALUATION_CONFIGS = {
    "including_misgendering": {
        "label_col": "label",
        "output_suffix": "including_misgendering",
        "longformer_models": [
            {
                "name": "Longformer_MIMIC",
                "model_dir": str(MODEL_DIR / 'mimic_no_preprocess_no_glob_binary'),
            },
            {
                "name": "Longformer_Berkeley_MIMIC",
                "model_dir": str(MODEL_DIR / 'new_berkeley_pretrain_to_mimic_v010725'),
            },
            {
                "name": "Longformer_Berkeley_Phenotype_MIMIC",
                "model_dir": str(MODEL_DIR / 'new_berkeley_pretrain_to_phenotype_to_mimic_v030325'),
            },
            {
                "name": "Longformer_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_GEP_Base_80_20'),
            },
            {
                "name": "Longformer_mimic_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_mimic_GEP_80_20'),
            },
            {
                "name": "Longformer_berkeley_mimic_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP_80_20'),
            },
            {
                "name": "Longformer_berkeley_phenotype_mimic_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_berkeley_phenotype_mimic_GEP_80_20'),
            },
        ],
        "bert_models": [
            {
                "name": "BERT_base",
                "model_dir": str(MODEL_DIR / 'BERT_GEP'),
            },
            {
                "name": "ClinicalBERT",
                "model_dir": str(MODEL_DIR / 'clinicalbert_GEP'),
            },
        ],
        "traditional_model_dir": str(MODEL_DIR / 'GEP'),
        "traditional_models": {
            "SVM_tfidf": "SVM_tfidf_best_model_f1_lowercase.joblib",
            "RF_count": "RF_count_best_model_f1_lowercase.joblib",
            "LR_tfidf": "LR_tfidf_best_model_f1_lowercase.joblib",
            "NB_tfidf": "NB_tfidf_best_model_f1_lowercase.joblib",
        },
    },

    "excluding_misgendering": {
        "label_col": "label_exclude_misgendering",
        "output_suffix": "excluding_misgendering",
        "longformer_models": [
            {
                "name": "Longformer_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_GEP_exclude_misgender'),
            },
            {
                "name": "Longformer_mimic_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_mimic_GEP_exclude_misgender'),
            },
            {
                "name": "Longformer_berkeley_mimic_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP_exclude_misgender'),
            },
            {
                "name": "Longformer_berkeley_phenotype_mimic_GEP",
                "model_dir": str(MODEL_DIR / 'Longformer_berkeley_phenotype_mimic_GEP_80_20_exclude_misgender'),
            },
        ],
        "bert_models": [
            {
                "name": "BERT_base",
                "model_dir": str(MODEL_DIR / 'BERT_GEP_exclude_misgender'),
            },
            {
                "name": "ClinicalBERT",
                "model_dir": str(MODEL_DIR / 'clinicalbert_GEP_exclude_misgender'),
            },
        ],
        "traditional_model_dir": str(MODEL_DIR / 'GEP'),
        "traditional_models": {
            "SVM_tfidf": "SVM_tfidf_best_model_f1_lowercase_exclude_misgender.joblib",
            "RF_count": "RF_count_best_model_f1_lowercase_exclude_misgender.joblib",
            "LR_tfidf": "LR_tfidf_best_model_f1_lowercase_exclude_misgender.joblib",
            "NB_tfidf": "NB_tfidf_best_model_f1_lowercase_exclude_misgender.joblib",
        },
    },
}

# ============================================================
# DATASETS
# ============================================================

class LongformerTextDataset(Dataset):
    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=4096,
        use_global_attention=True,
        global_attention_target="cls",
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_global_attention = use_global_attention
        self.global_attention_target = global_attention_target

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.texts[idx],
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
        )

        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)

        output = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(float(self.labels[idx]), dtype=torch.float),
        }

        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)
            if self.global_attention_target == "cls":
                global_attention_mask[0] = 1
            output["global_attention_mask"] = global_attention_mask

        return output


class ChunkedTextDataset(Dataset):
    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        chunk_size=510,
        max_length=512,
        doc_max_length=4096,
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length

        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.sep_token

    def __len__(self):
        return len(self.texts)

    def _chunk_ids(self, text):
        token_ids = self.tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False,
        )
        token_ids = token_ids[: self.doc_max_length]

        chunks = []
        for i in range(0, len(token_ids), self.chunk_size):
            core = token_ids[i : i + self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + core + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            else:
                chunk = chunk[: self.max_length]
            chunks.append(chunk)

        if not chunks:
            chunk = [self.tokenizer.cls_token_id, self.tokenizer.sep_token_id]
            chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunks = [chunk]

        return torch.tensor(chunks, dtype=torch.long)

    def __getitem__(self, idx):
        chunks = self._chunk_ids(self.texts[idx])
        return {
            "chunks": chunks,
            "label": torch.tensor(float(self.labels[idx]), dtype=torch.float),
            "num_chunks": chunks.size(0),
        }


def bert_collate_fn(batch):
    all_chunks = [b["chunks"] for b in batch]
    labels = torch.tensor([b["label"].item() for b in batch], dtype=torch.float)
    num_chunks = [b["num_chunks"] for b in batch]
    flat_chunks = torch.cat(all_chunks, dim=0)

    return {
        "chunks": flat_chunks,
        "labels": labels,
        "num_chunks": num_chunks,
    }

# ============================================================
# METRICS
# ============================================================

def compute_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
        "Positive_Prevalence": float(np.mean(y_true)),
        "N_Positive": int(np.sum(y_true)),
        "N_Negative": int(len(y_true) - np.sum(y_true)),
    }

    if len(np.unique(y_true)) > 1:
        metrics["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        metrics["PR_AUC"] = average_precision_score(y_true, y_prob)

    return metrics


def add_metric_rows(all_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_vec):
    groups = {
        "Overall": np.arange(len(y_true)),
        "GEP=0": np.where(gep_vec == 0)[0],
        "GEP=1": np.where(gep_vec == 1)[0],
    }

    for group_name, idx in groups.items():
        if len(idx) == 0:
            row = {
                "Outcome": outcome_name,
                "Model": model_name,
                "Group": group_name,
                "N": 0,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "F1": np.nan,
                "ROC_AUC": np.nan,
                "PR_AUC": np.nan,
                "Positive_Prevalence": np.nan,
                "N_Positive": 0,
                "N_Negative": 0,
            }
        else:
            metric_dict = compute_metrics(y_true[idx], y_pred[idx], y_prob[idx])
            row = {
                "Outcome": outcome_name,
                "Model": model_name,
                "Group": group_name,
                "N": int(len(idx)),
                **metric_dict,
            }

        all_rows.append(row)


def add_prediction_rows(all_pred_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_vec):
    for i in range(len(y_true)):
        all_pred_rows.append({
            "Outcome": outcome_name,
            "Model": model_name,
            "Row": int(i),
            "True": int(y_true[i]),
            "Pred": int(y_pred[i]),
            "Prob": float(y_prob[i]),
            "GEP": int(gep_vec[i]),
        })


# ============================================================
# PREDICTION HELPERS
# ============================================================

@torch.inference_mode()
def predict_longformer(model, data_loader):
    model.eval()
    all_probs = []
    all_preds = []

    for batch in tqdm(data_loader, desc="Longformer predicting", unit="batch", leave=True):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)

        global_attention_mask = batch.get("global_attention_mask", None)
        if global_attention_mask is not None:
            global_attention_mask = global_attention_mask.to(device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
        )

        probs = torch.sigmoid(outputs.logits).view(-1).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

    return np.asarray(all_preds), np.asarray(all_probs)


@torch.inference_mode()
def predict_bertlike(model, tokenizer, texts, labels):
    dataset = ChunkedTextDataset(
        texts=texts,
        labels=labels,
        tokenizer=tokenizer,
        chunk_size=CHUNK_SIZE,
        max_length=MAX_LENGTH,
        doc_max_length=DOC_MAX_TOKENS,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_BERT,
        shuffle=False,
        collate_fn=bert_collate_fn,
        pin_memory=torch.cuda.is_available(),
    )

    all_probs = []
    all_preds = []

    for batch in tqdm(loader, desc="BERT-like predicting", unit="batch", leave=True):
        chunks = batch["chunks"].to(device, non_blocking=True)
        num_chunks = batch["num_chunks"]
        attention_mask = (chunks != tokenizer.pad_token_id).to(device, non_blocking=True)

        logits = model(input_ids=chunks, attention_mask=attention_mask).logits.squeeze(-1)

        pooled_logits = []
        start = 0
        for n_chunks in num_chunks:
            pooled_logits.append(torch.max(logits[start : start + n_chunks]))
            start += n_chunks

        pooled_logits = torch.stack(pooled_logits)
        probs = torch.sigmoid(pooled_logits).detach().cpu().numpy().reshape(-1)
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

    return np.asarray(all_preds), np.asarray(all_probs)


def safe_predict_scores(model, texts):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(texts)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(texts)
        return np.asarray(scores, dtype=float)
    preds = model.predict(texts)
    return np.asarray(preds, dtype=float)

# ============================================================
# EVALUATION
# ============================================================

all_rows = []
all_pred_rows = []

for outcome_name, cfg in EVALUATION_CONFIGS.items():
    print("\n" + "#" * 120)
    print(f"Evaluating outcome: {outcome_name}")
    print("#" * 120)

    label_col = cfg["label_col"]

    texts = mimic_test["text"].tolist()
    labels = mimic_test[label_col].astype(int).to_numpy()
    gep_vec = mimic_test["GEP"].astype(int).to_numpy()

    # ----------------------------
    # Longformer models
    # ----------------------------
    for model_cfg in cfg["longformer_models"]:
        model_name = model_cfg["name"]
        model_dir = model_cfg["model_dir"]

        print(f"\nLoading Longformer model: {model_name}")

        try:
            tokenizer = LongformerTokenizer.from_pretrained(model_dir)
            model = LongformerForSequenceClassification.from_pretrained(model_dir)
            model.to(device)
            model.eval()

            dataset = LongformerTextDataset(
                texts=texts,
                labels=labels,
                tokenizer=tokenizer,
                max_length=4096,
            )

            loader = DataLoader(
                dataset,
                batch_size=BATCH_SIZE_LONGFORMER,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )

            y_pred, y_prob = predict_longformer(model, loader)

            n = min(len(labels), len(y_pred), len(y_prob), len(gep_vec))
            y_true = labels[:n]
            y_pred = y_pred[:n]
            y_prob = y_prob[:n]
            gep_sub = gep_vec[:n]

            add_metric_rows(all_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_sub)
            add_prediction_rows(all_pred_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_sub)

        except Exception as e:
            print(f"[Error] Longformer model failed: {model_name}: {e}")
            all_rows.append({
                "Outcome": outcome_name,
                "Model": model_name,
                "Group": "Error",
                "N": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "F1": np.nan,
                "ROC_AUC": np.nan,
                "PR_AUC": np.nan,
                "Positive_Prevalence": np.nan,
                "N_Positive": np.nan,
                "N_Negative": np.nan,
                "Error": str(e),
            })

    # ----------------------------
    # BERT / ClinicalBERT
    # ----------------------------
    for model_cfg in cfg["bert_models"]:
        model_name = model_cfg["name"]
        model_dir = model_cfg["model_dir"]

        print(f"\nLoading BERT-like model: {model_name}")

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_dir)
            if tokenizer.pad_token_id is None:
                tokenizer.pad_token = tokenizer.sep_token

            model = AutoModelForSequenceClassification.from_pretrained(model_dir)
            model.to(device)
            model.eval()

            y_pred, y_prob = predict_bertlike(model, tokenizer, texts, labels)

            n = min(len(labels), len(y_pred), len(y_prob), len(gep_vec))
            y_true = labels[:n]
            y_pred = y_pred[:n]
            y_prob = y_prob[:n]
            gep_sub = gep_vec[:n]

            add_metric_rows(all_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_sub)
            add_prediction_rows(all_pred_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_sub)

        except Exception as e:
            print(f"[Error] BERT-like model failed: {model_name}: {e}")
            all_rows.append({
                "Outcome": outcome_name,
                "Model": model_name,
                "Group": "Error",
                "N": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "F1": np.nan,
                "ROC_AUC": np.nan,
                "PR_AUC": np.nan,
                "Positive_Prevalence": np.nan,
                "N_Positive": np.nan,
                "N_Negative": np.nan,
                "Error": str(e),
            })

    # ----------------------------
    # Traditional ML models
    # ----------------------------
    for model_name, file_name in cfg["traditional_models"].items():
        model_path = os.path.join(cfg["traditional_model_dir"], file_name)

        print(f"\nLoading traditional ML model: {model_name}")

        if not os.path.exists(model_path):
            print(f"[Warning] Missing file: {model_path}")
            continue

        try:
            model = joblib.load(model_path)

            y_true = labels
            y_pred = model.predict(texts)
            y_prob = safe_predict_scores(model, texts)

            n = min(len(y_true), len(y_pred), len(y_prob), len(gep_vec))
            y_true = y_true[:n]
            y_pred = np.asarray(y_pred[:n]).astype(int)
            y_prob = np.asarray(y_prob[:n]).astype(float)
            gep_sub = gep_vec[:n]

            add_metric_rows(all_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_sub)
            add_prediction_rows(all_pred_rows, outcome_name, model_name, y_true, y_pred, y_prob, gep_sub)

        except Exception as e:
            print(f"[Error] Traditional model failed: {model_name}: {e}")
            all_rows.append({
                "Outcome": outcome_name,
                "Model": model_name,
                "Group": "Error",
                "N": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "F1": np.nan,
                "ROC_AUC": np.nan,
                "PR_AUC": np.nan,
                "Positive_Prevalence": np.nan,
                "N_Positive": np.nan,
                "N_Negative": np.nan,
                "Error": str(e),
            })

# ============================================================
# SAVE OUTPUTS
# ============================================================

results_df = pd.DataFrame(all_rows)
preds_df = pd.DataFrame(all_pred_rows)

metric_cols = [
    "Outcome",
    "Model",
    "Group",
    "N",
    "N_Positive",
    "N_Negative",
    "Positive_Prevalence",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "Error",
]

for col in metric_cols:
    if col not in results_df.columns:
        results_df[col] = np.nan

results_df = results_df[metric_cols]

metrics_csv = os.path.join(RESULT_DIR, "model_performance_with_roc_auc_and_pr_auc.csv")
metrics_xlsx = os.path.join(RESULT_DIR, "model_performance_with_roc_auc_and_pr_auc.xlsx")
preds_csv = os.path.join(RESULT_DIR, "model_predictions_with_outcomes.csv")
preds_xlsx = os.path.join(RESULT_DIR, "model_predictions_with_outcomes.xlsx")

results_df.to_csv(metrics_csv, index=False)
results_df.to_excel(metrics_xlsx, index=False)
preds_df.to_csv(preds_csv, index=False)
preds_df.to_excel(preds_xlsx, index=False)

print("\n Saved outputs:")
print(metrics_csv)
print(metrics_xlsx)
print(preds_csv)
print(preds_xlsx)

try:
    from IPython.display import display
    display(results_df)
except Exception:
    print(results_df)

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(results_df)